In [89]:
from dotenv import load_dotenv
load_dotenv()

True

In [90]:
from pydantic import BaseModel,Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated,Literal

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent

In [91]:
class FLowState(BaseModel):
    question:str = Field(description="Used asked question")
    category:Literal['coding','google_search','weather'] = Field(default='google_search')
    answer:str = Field(default="")

In [92]:
class QuestionCategory(BaseModel):
    category:Literal['coding','google_search','weather'] = Field(default='google_search',description="question category")

In [93]:

llm = ChatGroq(model="openai/gpt-oss-20b")
memory = InMemorySaver()

In [94]:
def check_question_category(state:FLowState)->FLowState:
    st_llm = llm.with_structured_output(QuestionCategory)
    res = st_llm.invoke(f"I want to know category of my question, question is: {state.question}. If you are not sure then give category as 'google_search'")
    # print(res)
    state.category = res.category
    return state

In [95]:
# flow = FLowState(question = "What is the address of jain pch rock garden phase 3")
# check_question_category(flow)

In [96]:
def route(state:FLowState) -> Literal['coding','google_search','weather']:
    return state.category

In [97]:
search = GoogleSerperAPIWrapper()
agent = create_agent(
    model=llm,
    tools=[search.run],
    system_prompt='You are a agent and can search for any question on google',
    checkpointer=memory
)

In [98]:
def coding_node(state:FLowState) -> FLowState:
    print("coding node")
    res=llm.invoke(f"You are a coding expert, answer this: {state.question}")
    state.answer = res.content
    return state

def weather_node(state:FLowState) -> FLowState:
    print("weather node")
    return state

def google_search_node(state:FLowState) -> FLowState:
    print("google search node")
    res = agent.invoke({"messages":[{"role":"user","content":state.question}]},{"configurable":{"thread_id":"abc123"}})
    state.answer = res["messages"][-1].content
    return state  

In [101]:
graph = StateGraph(FLowState)

graph.add_node("check_question_category",check_question_category)
graph.add_node("coding",coding_node)
graph.add_node("weather",weather_node)
graph.add_node("google_search",google_search_node)

graph.add_edge(START,"check_question_category")
graph.add_conditional_edges("check_question_category",route)
graph.add_edge("coding",END)
graph.add_edge("google_search",END)
graph.add_edge("weather",END)

compliedGraph = graph.compile()

In [102]:
res = compliedGraph.invoke({"question":"Who is the PM of india"})
res

google search node


{'question': 'Who is the PM of india',
 'category': 'google_search',
 'answer': 'The current Prime Minister of India is **Narendra\u202fModi**. He has been in office since 26\u202fMay\u202f2014 and continues to serve as the head of the Indian government.'}